<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="30%"></a>
</p>


<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

# Docker Clients, Registries, and Platforms

The previous notebook established a local Docker connection for the exercises. Before running a container, we still need to identify the image we want, check that it supports our computer, and see what Docker resources are already present.

For example, a Duckietown image name may include both a software distribution and a processor architecture. Choosing an image intended for the robot does not necessarily make it suitable for your base station computer. Similarly, finding an image in Docker’s local storage does not mean a container is running from it.

In this notebook, you will read image references, identify the daemon’s platform, and distinguish available images from running and stopped containers.

Use the base-station terminal and local Docker connection verified [in the previous notebook](./2-docker-containers-images-and-safe-local-practice.ipynb). All commands below inspect configuration, resources, or help text; they do not create or remove resources.

## Docker client, daemon, and registry

Recall that the `docker` command is the **client**. For operations such as pulling images or listing containers, the client sends requests to the selected **daemon**, a background service that manages images, containers, networks, and volumes. The client and daemon can run on the same computer, but the Docker context and any connection overrides decide which daemon receives the request. 

<figure id="figure-1" class="lx-figure">
  <pre style="display:inline-block; margin:0; text-align:left;">
Docker client                 Local Docker daemon              Registry
     |                                |                            |
     | Request an image download      |                            |
     +-------------------------------&gt;| Request image content      |
                                      +---------------------------&gt;|
                                      |&lt;---------------------------+
                                      | Receive and store image
  </pre>
  <figcaption>
    Figure 1: The client requests the operation; the daemon downloads the image from the registry.
  </figcaption>
</figure>

A __registry__ stores images for download and publication. Docker Hub is a public registry. A repository groups related images, and a tag identifies a particular version or variant. The naming structure is illustrated in [Figure 2](#figure-2).

<figure id="figure-2" class="lx-figure">
  <pre style="display:inline-block; margin:0; text-align:left;">
    registry / repository : tag
    docker.io / duckietown/dd-linux-and-networking : ente-arm64v8
  </pre>
  <figcaption>
    Figure 2: Registry, repository, and tag structure for an image reference.
  </figcaption>
</figure>

The `docker.io` registry name is often omitted. Use explicit image tags, such as `duckietown/dd-linux-and-networking:ente-arm64v8`, to name the intended version or variant. The `latest` tag is an ordinary label maintained by the publisher; it does not automatically identify the newest version. Any tag can change. A **digest** (`repository@sha256:...`) identifies specific image content when exact reproducibility is required.

| Part | Value | Meaning |
| --- | --- | --- |
| Registry | `docker.io` | Docker Hub |
| Namespace | `duckietown` | Organization or account / namespace|
| Repository within the namespace | `dd-linux-and-networking` | Related images for this software |
| Tag | `ente-arm64v8` | A publisher-assigned version or variant label |

For example, when you ask Docker to download an image, the client sends the request to the selected daemon. The daemon obtains the image from the registry and stores it in its own image storage. Docker’s official [image-reference documentation](https://docs.docker.com/reference/cli/docker/image/tag/) offers additional information for the naming rules.

## Processor architecture and platform

An image may contain programs with central processing unit (CPU)-specific instructions. A typical laptop might use the `linux/amd64` platform, while a physical Duckiedrone DD24 uses `linux/arm64`. An image built only for one platform might not run on another.

You can check your daemon platform with:

```bash
docker version --format '{{.Server.Os}}/{{.Server.Arch}}'
```

For example, a Duckietown Workspace on an ARM64 system can report: 

```shell
linux/arm64
```

Many public images, including the `alpine:3.21` image used in this LX, a small Linux environment suitable for practicing container commands, are multi-platform. Docker normally chooses the variant that matches the computer running the daemon. Duckietown image tags can include an architecture label such as `arm64v8` or `amd64`, corresponding to Docker platforms `linux/arm64` and `linux/amd64`, respectively. An image built for a different architecture may fail unless suitable emulation is available. Use an image matching the target platform for these exercises.

## Inspecting images and containers

`docker image ls` lists locally cached images. Inspect the local image cache without creating, removing, or changing resources:

```bash
docker image ls
```

A typical output would look like:

```text
REPOSITORY   TAG    IMAGE ID       CREATED       SIZE
alpine       3.21   a1b2c3d4e5f6   2 weeks ago   8 MB
```

Similarly, `docker container ls` lists currently running containers, and `--all` includes stopped containers. 

Inspect both views of the local container inventory:

```bash
docker container ls
docker container ls --all
```

An illustrative result from `docker container ls --all` is:

```text
CONTAINER ID   IMAGE         STATUS                      NAMES
b2c3d4e5f6a7   alpine:3.21   Exited (0) 10 minutes ago   lx-docker-example
```

These commands can show resources belonging to other work on a shared base station, so treat names, paths, image tags, and container configuration as information to protect rather than material to copy into reports. In the example above, the container still exists, but its main process has finished. `Exited (0)` reports that the process ended with exit status zero. The container name, `lx-docker-example`, identifies this instance; `alpine:3.21` identifies the image reference used to create it.

The shorter command `docker ps` is an alias for `docker container ls`; `docker ps -a` includes stopped containers. See [`docker container ls`](https://docs.docker.com/reference/cli/docker/container/ls/).

It is not necessary to memorize all commands, but you should know how to find the help text for any command. For example, `docker help` lists the main command groups, and `docker container help` lists the subcommands for managing containers, while `--help` provides usage information for specific commands. Try the following commands in the base-station terminal:

```bash
docker help
docker container ls --help
docker image ls --help
```

## Further reading

See Docker's [guide to Docker contexts](https://docs.docker.com/engine/manage-resources/contexts/) for selecting a daemon other than the authorized local target.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()


## Summary and conclusions

An image reference identifies a registry, repository, and tag or digest. The image must support the daemon’s platform, and listing images is distinct from listing running or stopped containers. Next, [review container access and exercise boundaries](./4-docker-security-and-duckiedrone-boundaries.ipynb) before creating resources.
